In [9]:
######################################################################################################################################
import json,re,os
from openai import OpenAI  # your OpenRouter OpenAI wrapper
import numpy as np
from neo4j import GraphDatabase
from general_tools import call_llm,get_XML_context,extract_paper_basic_info,get_XML_titleabstract#


In [ ]:
# ---------- config------------------
api = "your LLM api"

config_list1 = [{
    'model': 'LLM model',#'
    'base_url': 'LLM url',
}]#

config_list2 = [{
    'model': 'LLM model',#'
    'base_url': 'LLM url',
}]#

config_list3 = [{
    'model': 'LLM model',
    'base_url': 'LLM url',
}]


llm_config1 = {
    'seed': 42,
    'config_list': config_list1,
    'temperature': 0
}

llm_config2 = {
    'seed': 42,
    'config_list': config_list2,
    'temperature': 0
}

llm_config3 = {
    'seed': 42,
    'config_list': config_list3,
    'temperature': 0
}

NEO4J_URI = "NEO4J url"  
NEO4J_USER = "your neo4j username"
NEO4J_PASS = "your neo4j password"

In [3]:
# ---------- LLM composition + conditions extractor ----------
def extract_composition_conditions(fulltext_context="", api=api,config_list=config_list1, llm_config=llm_config1):
    prompt = f"""
You are an expert materials scientist.

Input:
Full paper context: {fulltext_context}.

Task:
From the input paper context, extract **all the sample compositions that were actually studied and tested in THIS paper**.

Extraction Rules:
1. **Include all experimentally studied compositions** reported in this paper. Exclude compositions that are only referenced from other works.
2. Include sample labels. If no label exists in text, assign a unique one (e.g., Sample_1, Sample_2, etc.).
3. Always include the full composition string as written in the paper under `"composition_string"`. Preserve the composition EXACTLY as written in the text (order, numbers, ranges, x-variables, etc.).
4. Additionally, parse the composition into `"composition_elements"`, a list of dictionaries with `"element"`, `"amount"` (numeric), and `"text"` (original string from text). 
5. In `"composition_elements"`:
    - Always output the **full composition with explicit numeric values** for every element.
    - Numbers must be floats rounded to 5 decimal place.
    - Element symbols must be standard chemical symbols (e.g., Ni, Co, Cr).
    - If an element is the **balance element**, calculate its value as `100 - sum(all other numeric amounts)` and fill in `"amount"`.
    - If no number is given or it is a trace amount(e.g., "0.001 ppm"), store `"text"` as in the paper and `"amount"` as 0.
    - If a range is given (e.g., 20–25at% Al), store the range as `"text"` and `"amount"` as the midpoint.

STRICT RESPONSE RULES:
- Respond ONLY with valid JSON.
- Do NOT include text, commentary, markdown code fences, or explanations before or after the JSON.
- The JSON must be a **single JSON array** (a list `[...]` of objects), where each object represents one sample entry.
- Do NOT return multiple top-level objects or extra text.
- Ensure the JSON is valid and parsable by Python json.loads().

Return JSON with this format:
[
    {{
    "sample_label": "assigned or existing label",
    "composition_string": "composition string as in paper",
    "composition_elements": [
        {{
            "element": "Ni",
            "amount": float or 0 if trace/balance,
            "text": "original string from text"
        }},
        {{
            "element": "Co",
            "amount": float or 0 if trace/balance,
            "text": "original string from text"
        }}
        // include all elements in this way
    ],
    "composition unit": "at% or wt%",
    }}
]
"""

##############################################################################
#Respond only with JSON.
    print("\n=== LLM Prompt ===")
    print(prompt)
##############################################################################
    response = call_llm(prompt, api, config_list, llm_config)
    reply = response.choices[0].message.content.strip()

    # Clean markdown/code fences if they exist
    cleanedreply = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.MULTILINE).strip()
##############################################################################
    print("=== LLM Response ===")
    print(cleanedreply)
    print("====================")
    return cleanedreply

In [4]:
# ---------- LLM composition + conditions extractor ----------
def extract_domain(fulltext_context="",api=api, config_list=config_list2, llm_config=llm_config2):
    prompt = f"""
You are an expert materials scientist.

Task:
From the following research paper text, identify which scientific domains it primarily belongs to. 
Example domains are ["Oxidation", "Corrosion", "Mechanical"], 
but if none of these clearly apply, you may add one or more other relevant domains.

Paper context:
{fulltext_context}

Extraction Rules:
1. The paper may belong to **more than one domain**. Always list all that apply.
2. Base your judgment only on the paper content, not on generic expectations.
3. Use consistent capitalization (e.g., "Oxidation" not "oxidation").

STRICT RESPONSE RULES:
- Respond **only** with valid JSON.
- Do **not** include commentary, markdown fences, or explanations.
- The JSON must be a **single object** of the form:
  {{
    "domain_tags": ["...", "..."]
  }}
- Ensure the JSON is fully valid and parsable by `json.loads()`.
"""

##############################################################################
#Respond only with JSON.
    print("\n=== LLM Prompt ===")
    print(prompt)
##############################################################################
    response = call_llm(prompt, api, config_list, llm_config)

    reply = response.choices[0].message.content.strip()
    # Clean markdown/code fences if they exist
    cleanedreply = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.MULTILINE).strip()
##############################################################################
    print("=== LLM Response ===")
    print(cleanedreply)
    print("====================")
    
    return cleanedreply

In [5]:

######################################################################################################################################
# ==== HELPER: LLM classification ====
def causal_extraction(material_info,fulltext_context="", api=api,config_list=config_list3, llm_config=llm_config3):#
    prompt = f"""
You are a materials science knowledge extraction assistant.  
Your task is to build a **Causal Map** that connects **Composition–Processing–Structure–Property–Mechanism** relationships described in a scientific paper.

### Inputs
1. **Paper context (text)**:
{fulltext_context}

2. **Composition to analyze (only analyze this composition)**:
{material_info}

### Task
Generate a structured JSON that represents the **causal relationships** for the given composition.  
The JSON must contain:
- **nodes**: Each unique entity (processing step, structure, or property).  
- **edges**: Directed causal links between nodes (e.g., processing → structure, structure → property), with optional mechanisms explaining the connection.

### Rules for Extraction
###1.Node Definition
Each node must include:
    - `id`: unique label following these patterns:
    - Processing → `"PRC1"`, `"PRC2"`, ...
    - Structure → `"S1"`, `"S2"`, ...
    - Property → `"P1"`, `"P2"`, ...
    - `type`: one of `["processing", "structure", "property"]`
    - `description`: a list of dictionaries, each with **a concise scientific phrase** directly extracted or paraphrased faithfully from the paper (no creativity).
    
#### 2. Edge Definition
Each edge must include:
    - `source`: the cause node id.
    - `target`: the effect node id.
    - `via_mechanism`: list of dictionaries containing:
        - `name`: short name of the mechanism (e.g., "oxide scale growth").
        - `description`: how this mechanism explains the causal link.

#### 3. Cause–Effect Direction Rules
Follow scientifically valid causal directions:
    - Processing → Structure (e.g., “heat treatment → precipitate formation”)
    - Structure → Property (e.g., “fine grains → improved oxidation resistance”)    
    - Mechanisms should describe the underlying link (e.g., “grain boundary diffusion controls oxide growth”).

### Additional Instructions
    - Only include scientifically valid and well-supported causal relationships.
    - Avoid speculative or unclear links.
    - Use concise, formal materials-science phrasing.
    - If multiple causal chains exist, include all (each as a separate edge).
    - Respond **only** with valid JSON — no markdown, explanations, or commentary.
    - Ensure the JSON is **fully valid and parsable** by `json.loads()` in Python.
 
Return JSON with this format:
### Output Format (JSON)
{{
  "nodes": [
    {{
      "id": "PRC1",
      "type": "processing",
      "description": "heat-treated at 1200°C for 1h",
    }},
    {{
      "id": "S1",
      "type": "structure",
      "description": "γ′ precipitates",
    }}
  ],
  "edges": [
    {{
      "source": "PRC1",
      "target": "S1",
      "via_mechanism": [
        {{
          "name": "γ′ strengthening",
          "description": "γ′ phase hinders dislocation motion",
        }}
      ]
    }}
  ]
}}
"""    
    print("\n=== LLM Prompt ===")
    print(prompt)

    response = call_llm(prompt, api, config_list, llm_config)
    reply = response.choices[0].message.content.strip()

    # Clean markdown/code fences if they exist
    cleanedreply = re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.MULTILINE).strip()
##############################################################################
    print("=== LLM Response ===")
    print(cleanedreply)
    print("====================")
    return cleanedreply

In [6]:
pdf_dir = "papers"
pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]
pdf_files.sort()  # ensures deterministic order

In [ ]:
for pdf_name in pdf_files:
    # ==== CONFIG ====
    GROBID_XML_FOLDER = "grobid_xml_outputs"
    OUTPUT_JSON_PATH = os.path.join("text_result",str(pdf_name.split(".")[0])+"_text_results.json")
    XML_PATH = os.path.join(GROBID_XML_FOLDER,str(pdf_name.split(".")[0])+".xml")
    os.makedirs('text_result', exist_ok=True)
##############################################################################
    metadata = extract_paper_basic_info(XML_PATH)
    print("\n=== Paper Metadata ===")
    print(f"File name: {metadata.get('filename', 'N/A')}")
    print(f"DOI: {metadata.get('doi', 'N/A')}")
    print(f"Title: {metadata.get('title', 'N/A')}")
    print(f"journal: {metadata.get('journal', 'N/A')}")
    print(f"year: {metadata.get('year', 'N/A')}")
    print("Authors:", ", ".join(metadata.get("authors", [])))
    print("======================\n")
    full_context = get_XML_titleabstract(XML_PATH)
    print(f"paper length: {len(full_context)} chars")
    print(f"paper title and abstract: {full_context} chars")
    domain_str = extract_domain(full_context)
    print(domain_str)

    domain_dict = json.loads(domain_str)
    metadata["domain_tags"] = domain_dict.get("domain_tags", [])
##################################################################################################
    full_context = get_XML_context(XML_PATH)
    print(f"paper length: {len(full_context)} chars")

    text_str = extract_composition_conditions(full_context)
    print(text_str)
###################################################################################################################################################
    output = {}
    classification_dict = json.loads(text_str)
###############################################################################
    # Save progress incrementally
    output = [{
        "paper": metadata,
        "samples": classification_dict
    }]

    # Save updated JSON
    with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"✅ Updated JSON saved to {OUTPUT_JSON_PATH}") 



=== Paper Metadata ===
File name: 000001.xml
DOI: 10.1016/j.jallcom.2015.09.042
Title: The evolution of microstructures and high temperature properties of AlxCo1.5CrFeNi1.5Tiy high entropy alloys
journal: Journal of Alloys and Compounds
year: 2015-09-09
Authors: Yao-Jen Chang, An-Chou Yeh, K Tsai, M Tsai, J Yeh, F Otto, A Dlouhý, C Somsen, H Bei, G Eggeler, E George, E Huang, D Yu, J Yeh, C Lee, K An, S Tu, Fig, C Tung, J Yeh, T Shun, S Chen, Y Huang, H Chen, S Singh, N Wanderka, B Murty, U Glatzel, J Banhart, C Juan, C Hsu, C Tsai, W Wang, T Sheu, J Yeh, S Chen, O Senkov, G Wilks, D Miracle, C Chuang, P Liaw, O Senkov, G Wilks, J Scott, D Miracle, O Senkov, J Scott, S Senkova, D Miracle, C Woodward, O Senkov, S Senkova, C Woodward, O Senkov, J Scott, S Senkova, F Meisenkothen, D Miracle, C Woodward, M Chuang, M Tsai, W Wang, S Lin, J Yeh, C Hsu, J Yeh, S Chen, T Shun, S Chen, W Tang, Y Kuo, S Chen, C Tsau, T Shun, J Yeh, Y Chou, Y Wang, J Yeh, H Shih, C Lee, C Chang, Y Chen, J Yeh, H

KeyboardInterrupt: 

In [8]:
for pdf_name in pdf_files:
    # pdf_name = str(i)
    GROBID_XML_FOLDER = "grobid_xml_outputs"
    XML_PATH = os.path.join(GROBID_XML_FOLDER,str(pdf_name.split(".")[0])+".xml")
    OUTPUT_JSON_PATH = os.path.join("text_result",str(pdf_name.split(".")[0])+"_text_results_with_causal.json")
    MATERIAL_JSON_PATH = os.path.join("text_result",str(pdf_name.split(".")[0])+"_text_results.json")
##############################################################################
    with open(MATERIAL_JSON_PATH, "r", encoding="utf-8") as f:
        material_entries = json.load(f)

    updated_entries = []
    paper_metadata = material_entries[0]['paper']

    for index, entry in enumerate(material_entries[0]['samples']):
##############################################################################
#################prepare full paper input###################################################
        extra_context = get_XML_context(XML_PATH)
    #################prepare material input###################################################
        material_info = []
        material_info.append({
            "sample_label": entry["sample_label"],
            "composition_string": entry["composition_string"]
        })
        material_info = json.dumps(material_info, indent=2)
# #######################################################       
        classification_str  = causal_extraction(material_info,extra_context)
        try:
            classification_dict = json.loads(classification_str)
        except json.JSONDecodeError:
            classification_dict = {classification_str
            }
        # Merge classification into the entry
        entry.update(classification_dict)
        updated_entries.append(entry)
###############################################################################
        # Save progress incrementally
        output_json = [{
            "paper": paper_metadata,
            "samples": updated_entries
        }]

        # Save updated JSON
        with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(output_json, f, indent=2, ensure_ascii=False)
        print(f"✅ Updated {entry['sample_label']} saved to {OUTPUT_JSON_PATH}")

    print(f"🎯 Finished processing paper {pdf_name}")


=== LLM Prompt ===

You are a materials science knowledge extraction assistant.  
Your task is to build a **Causal Map** that connects **Composition–Processing–Structure–Property–Mechanism** relationships described in a scientific paper.

### Inputs
1. **Paper context (text)**:
The evolution of microstructures and high temperature properties of AlxCo1.5CrFeNi1.5Tiy high entropy alloys Ministry of Science and Technology, Taiwan . Elsevier BV Copyright Elsevier BV 9 September 2015 Yao-Jen Chang Department of Materials Science and Engineering National Tsing Hua University 30013 Hsinchu Taiwan An-Chou Yeh Department of Materials Science and Engineering National Tsing Hua University 30013 Hsinchu Taiwan The evolution of microstructures and high temperature properties of AlxCo1.5CrFeNi1.5Tiy high entropy alloys Journal of Alloys and Compounds Journal of Alloys and Compounds 0925-8388 Elsevier BV 653 9 September 2015 D2F1603119DDFDF23181E97A9BF4F051 10.1016/j.jallcom.2015.09.042 Received 26 

KeyboardInterrupt: 

In [ ]:
# ======================================
# Helper Function
# ======================================
def normalize_composition(comp_elements):
    """
    Generate a canonical composition key based on elements and their numeric amounts,
    ignoring order and text formatting.
    """
    if not comp_elements:
        return "unknown"
    
    comp_dict = {}
    for e in comp_elements:
        try:
            el = e.get("element")
            amt = float(e.get("amount", 0))
            if el:
                comp_dict[el] = amt
        except (ValueError, TypeError):
            continue

    normalized = "-".join([f"{el}{comp_dict[el]}" for el in sorted(comp_dict.keys())])
    return normalized

Name = ['Al','Co','Cr','Ni','Fe','Si','Y','Hf','Nb','Mn','C','Mo','B','Zr','W',
'Ti','Ta','V','S','C','Cu','N','Ce','P','Re','La','Sn','Mg','Sc','O','Dy','Pt','Y2O3','Th','Ru','H',
'Na','Ca','Cr2O3','SiO2','Al2O3','TiO2','La2O3','Eu2O3','CeO2','Sm2O3','Gd2O3','MgO','ZrO2','Ta2O5',
'TiO2','Yb2O3','Dy2O3','Lu2O3','Er2O3','Ho2O3','Tb2O3','Gd','Pd','Sm','Yb','Nd']

Atomic_weight = [26.9815,58.9331,51.9961,58.6934,55.845,28.0855,88.905,178.49,
92.906,54.9380,12.0107,95.94,10.811,91.224,183.84,47.867,180.947,50.9415,32.065,
12.0107,63.546,14.0067,140.116,30.9737,186.207,138.905,118.710,24.3050,44.9559,15.9994,
162.5,195.084,225.8102,232.038,101.07,1.00784,22.9897692819791,40.078,151.9904,60.0838,
101.9612,79.8655,325.8092,351.927,172.1145,348.7276,362.5024,40.3045,123.2224,262.4442,
79.8655,394.1064,372.9972,397.9318,382.5164,377.8588,365.849,157.2521,106.4153,150.3647,173.0541,144.2416]

Atomic_weight = np.array(Atomic_weight)

# ===============================================================
def wt_to_at_percent(elements, wt_values):
    idx = []

    for el in elements:
        if el not in Name:
            print(f"Unknown element: {el} → cannot convert.")
            return None
        idx.append(Name.index(el))

    aw = Atomic_weight[idx]
    wt_array = np.array(wt_values, dtype=float)

    at = (wt_array / aw) / np.sum(wt_array / aw) * 100
    return at.tolist()

# ===============================================================
def process_json(data):
    # for entry in data:
    for sample in data["samples"]:
        unit = sample.get("composition unit", "").strip()

        elements = [e["element"] for e in sample["composition_elements"]]
        amounts = [e["amount"] for e in sample["composition_elements"]]
        name  = sample.get("sample_label","")
        # if unit == "at%":
        if "at" in unit:
            print(f"✅{name} composition unit is at% no need to conver")
            # Already correct
            continue

        # elif unit == "wt%":
        elif "wt" in unit:
            print(f"🧹{name} composition unit is wt% conver to at%")
            at_percent = wt_to_at_percent(elements, amounts)
            if at_percent is None:
                print(f"{name} conversion error")
                continue

            # Update JSON values
            for i, at in enumerate(at_percent):
                sample["composition_elements"][i]["amount"] = round(at, 2)

            sample["composition unit"] = "at%"  # update unit

        else:
            print(f"🎯{name} has Unknown unit: {unit}")
            continue

    return data

###Add composition embedding to KG
ELEMENTS = ['Al','Co','Cr','Ni','Fe','Si','Y','Hf','Nb','Mn','C','Mo','B','Zr','W',
'Ti','Ta','V','S','C','Cu','N','Ce','P','Re','La','Sn','Mg','Sc','O','Dy','Pt','Y2O3','Th','Ru','H',
'Na','Ca','Cr2O3','SiO2','Al2O3','TiO2','La2O3','Eu2O3','CeO2','Sm2O3','Gd2O3','MgO','ZrO2','Ta2O5',
'TiO2','Yb2O3','Dy2O3','Lu2O3','Er2O3','Ho2O3','Tb2O3','Gd','Pd','Sm','Yb','Nd']
def compkey_to_vector(comp_key, elements=ELEMENTS):
    """
    comp_key: string like "Al24.0-Co16.0-Cr22.0-Fe8.0-Ni30.0"
    returns: list of floats in ELEMENTS order
    """
    vec = []
    comp_dict = {}
    for part in comp_key.split('-'):
        elem = ''.join([c for c in part if c.isalpha()])
        amt = ''.join([c for c in part if c.isdigit() or c=='.'])
        if elem and amt:
            comp_dict[elem] = float(amt)
    for e in elements:
        vec.append(comp_dict.get(e, 0.0))
    return vec

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

json_dir = "text_result"
json_files = [f for f in os.listdir(json_dir) if f.lower().endswith("with_causal.json")]
json_files.sort()  # ensures deterministic order
print(f"📂 Found {len(json_files)} JSON files to import from: {json_dir}")

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("🧹 Cleared existing Neo4j database.")

    for filename in json_files:
        json_path = os.path.join(json_dir, filename)
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"⚠️ Error reading {filename}: {e}")
            continue
        
####################patch: conver wt to at, dont not change the initial json file#########################################
        data = [process_json(data[0])]
######################################################################
        print(f"📘 Importing {filename} ...")
        paper_info = data[0]["paper"]
        doi = paper_info.get("doi", "")
        title = paper_info.get("title", "")
        year = paper_info.get("year", "")
        filename = paper_info.get("filename", "")
        paper_domains = paper_info.get("domain_tags", [])

        # Create paper node
        session.run(
            """
            MERGE (p:Paper {title:$title})
            SET p.doi=$doi, p.year=$year, p.filename=$filename, p.domain_tags=$domain_tags
            """,
            title=title, doi=doi, year=year, filename=filename, domain_tags=paper_domains
        )

        for sample in data[0].get("samples", []):
            sample_label = sample.get("sample_label", "")
            comp_str = sample.get("composition_string", "")
            comp_elems = sample.get("composition_elements", [])
            comp_key = normalize_composition(comp_elems)

            # Merge Composition node
            session.run(
                """
                MERGE (c:Composition {key:$key})
                SET c.composition_elements=$elems
                """,
                key=comp_key,
                elems=json.dumps(comp_elems)
            )

            # Merge Sample node (by composition only)
            session.run(
                """
                MERGE (s:Sample {composition_key:$key})
                ON CREATE SET s.composition_string=$comp_str, s.labels=[$label]
                ON MATCH SET s.labels = coalesce(s.labels, []) + $label
                WITH s
                MATCH (p:Paper {title:$title})
                MERGE (p)-[:HAS_SAMPLE]->(s)
                WITH s
                MATCH (c:Composition {key:$key})
                MERGE (s)-[:BELONGS_TO_COMPOSITION]->(c)
                """,
                key=comp_key,
                comp_str=comp_str,
                label=sample_label,
                title=title
            )

            # Create paper-specific Nodes
            for node in sample.get("nodes", []):
                session.run(
                    """
                    MERGE (n:Node {id:$id, composition_key:$key, paper_title:$title})
                    SET n.type=$type, n.description=$desc
                    WITH n
                    MATCH (s:Sample {composition_key:$key})
                    MERGE (s)-[:HAS_NODE]->(n)
                    """,
                    id=node.get("id", ""),
                    key=comp_key,
                    type=node.get("type", ""),
                    desc=json.dumps(node.get("description", [])),
                    title=title
                )

            # Create paper-specific edges
            for edge in sample.get("edges", []):
                src = edge.get("source", "")
                tgt = edge.get("target", "")
                for mech in edge.get("via_mechanism", []):
                    session.run(
                        """
                        MATCH (a:Node {id:$src, composition_key:$key, paper_title:$title}),
                              (b:Node {id:$tgt, composition_key:$key, paper_title:$title})
                        MERGE (a)-[r:VIA_MECHANISM {name:$name, paper_title:$title}]->(b)
                        SET r.description=$desc
                        """,
                        src=src,
                        tgt=tgt,
                        key=comp_key,
                        name=mech.get("name", ""),
                        desc=mech.get("description", ""),
                        title=title
                    )

    driver.close()
    print("✅ All JSON papers imported successfully!")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
with driver.session() as session:
    all_nodes = session.run("MATCH (s:Sample) RETURN s.composition_key AS comp_key")
    for record in all_nodes:
        comp_key = record["comp_key"]
        emb = compkey_to_vector(comp_key)
        session.run(
            "MATCH (s:Sample {composition_key: $comp_key}) "
            "SET s.embedding = $embedding",
            {"comp_key": comp_key, "embedding": emb}
        )